# Instructor Solution: Student Selenium Exercise

This solution notebook follows the same order and structure as `day_3_selenium_student_exercise.ipynb`.

Each task appears in the same sequence as the student notebook, but the TODO code is completed.

In [ ]:
from pathlib import Path
from urllib.parse import quote
import time

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

outdir = Path("data")
raw_dir = outdir / "raw"
processed_dir = outdir / "processed"
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)


## The Practice Page

Same local practice page as the student exercise. No live website is contacted.

In [ ]:
exercise_html = """
<!doctype html>
<html>
<head>
  <title>Research Task Selenium Exercise</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 32px; line-height: 1.4; }
    .control-panel { border: 2px solid #555; padding: 16px; margin-bottom: 20px; max-width: 820px; }
    input { font-size: 18px; padding: 7px; width: 360px; }
    button { font-size: 17px; padding: 8px 12px; margin: 6px; cursor: pointer; }
    #message { background: #eef; padding: 10px; font-weight: bold; }
    .result-card { border: 1px solid #888; padding: 12px; margin: 10px 0; max-width: 650px; background: #fafafa; }
    .result-card a { font-weight: bold; }
    .scroll-zone { height: 850px; background: linear-gradient(#fff, #eee); padding-top: 30px; color: #555; }
    #evidence-end { border-top: 4px solid #444; padding-top: 16px; }
  </style>
</head>
<body>
  <h1>Research Task Exercise Page</h1>
  <p>This local page behaves like a tiny interactive research-task tracker.</p>

  <div class="control-panel">
    <label for="query-box">Search keyword</label><br>
    <input id="query-box" name="keyword" placeholder="enter a keyword">
    <button id="run-filter">Run filter</button>
    <button id="create-result">Create result</button>
    <p id="message">No keyword submitted yet.</p>
  </div>

  <section id="result-list" aria-label="research results">
    <article class="result-card" data-record-id="result-1" data-keyword="governance">
      <a class="item-link" href="/records/1">Governance reading note</a>
      <p>Initial visible result.</p>
    </article>
  </section>

  <div class="scroll-zone">Scroll down after creating results.</div>

  <h2 id="evidence-end">Evidence Checkpoint</h2>
  <p>This is where you should scroll before taking a screenshot.</p>

  <script>
    let resultCounter = 1;

    document.querySelector('#run-filter').addEventListener('click', function() {
      const keyword = document.querySelector('#query-box').value;
      document.querySelector('#message').textContent = 'Active keyword: ' + keyword;
    });

    document.querySelector('#query-box').addEventListener('keydown', function(event) {
      if (event.key === 'Enter') {
        document.querySelector('#run-filter').click();
      }
    });

    document.querySelector('#create-result').addEventListener('click', function() {
      resultCounter = resultCounter + 1;
      const keyword = document.querySelector('#query-box').value || 'unlabeled';
      const card = document.createElement('article');
      card.className = 'result-card';
      card.setAttribute('data-record-id', 'result-' + resultCounter);
      card.setAttribute('data-keyword', keyword);
      card.innerHTML = '<a class="item-link" href="/records/' + resultCounter + '">Generated result ' + resultCounter + '</a>' +
                       '<p>Generated for keyword: ' + keyword + '</p>';
      document.querySelector('#result-list').appendChild(card);
    });
  </script>
</body>
</html>
"""

exercise_url = "data:text/html;charset=utf-8," + quote(exercise_html)
print(exercise_url[:120] + "...")


## Exercise Instructions

Solved version of the student workflow.

This solution completes the browser actions in order:

1. start Chrome and open `exercise_url`;
2. find the keyword input, type a keyword, and submit it;
3. wait until the message text changes;
4. click the Create result button several times;
5. extract result title text plus hidden `data-record-id` and `data-keyword` attributes;
6. scroll to the evidence checkpoint;
7. save a screenshot and a CSV table;
8. close the browser.


In [ ]:
# Task 1: Start Chrome and open the local exercise page.

options = Options()
options.add_argument("--window-size=1100,850")

# Selenium starts and controls a real browser, not just an HTTP request.
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

# Open the local page in the browser.
driver.get(exercise_url)

print("Opened:", driver.title)


## Task 2: Type and Submit a Keyword

Solution: use `#query-box` for the input. Submit by pressing Enter. Clicking `#run-filter` would also work.

In [ ]:
# Task 2: Type a keyword and submit it.

keyword = "platform accountability"

# Find the keyword input field.
keyword_input = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "#query-box"))
)

# Click into the input and type the keyword.
keyword_input.click()
keyword_input.send_keys(keyword)

# Submit the keyword by pressing Enter.
keyword_input.send_keys(Keys.ENTER)


## Task 3: Wait for Evidence That the Page Changed

Solution: wait until the message element `#message` contains the submitted keyword.

In [ ]:
# Task 3: Wait until the message text contains your keyword.

wait.until(
    EC.text_to_be_present_in_element((By.CSS_SELECTOR, "#message"), keyword)
)

message_text = driver.find_element(By.CSS_SELECTOR, "#message").text
print(message_text)


## Task 4: Click to Create Result Cards

Solution: use `#create-result` and click it three times.

In [ ]:
# Task 4: Click the Create result button several times.

number_of_clicks = 3

create_button = driver.find_element(By.CSS_SELECTOR, "#create-result")

for click_number in range(number_of_clicks):
    create_button.click()
    time.sleep(0.5)

print("Created result clicks:", number_of_clicks)


## Task 5: Extract Visible Text and Hidden Attributes

Solution: repeated records are `.result-card`; the title link is `a.item-link`; hidden attributes are `data-record-id` and `data-keyword`.

In [ ]:
# Task 5: Extract result-card data into a DataFrame.

result_rows = []

result_cards = driver.find_elements(By.CSS_SELECTOR, ".result-card")

for card in result_cards:
    title_link = card.find_element(By.CSS_SELECTOR, "a.item-link")
    result_rows.append(
        {
            "record_id": card.get_attribute("data-record-id"),
            "keyword": card.get_attribute("data-keyword"),
            "result_title": title_link.text,
            "link_href": title_link.get_attribute("href"),
        }
    )

results_df = pd.DataFrame(result_rows)
display(results_df)


## Task 6: Scroll and Save Evidence

Solution: scroll to `#evidence-end`, save a screenshot, and save `results_df` as CSV.

In [ ]:
# Task 6: Scroll to the evidence checkpoint and save outputs.

screenshot_path = raw_dir / "notebook_selenium_student_exercise_solution.png"
results_path = processed_dir / "notebook_selenium_student_results_solution.csv"

evidence_checkpoint = driver.find_element(By.CSS_SELECTOR, "#evidence-end")

driver.execute_script("arguments[0].scrollIntoView();", evidence_checkpoint)
time.sleep(1)

driver.get_screenshot_as_file(str(screenshot_path))
results_df.to_csv(results_path, index=False)

print("Saved screenshot:", screenshot_path)
print("Saved result table:", results_path)


## Task 7: Close the Browser

Solution: close the Selenium-controlled browser.

In [ ]:
# Task 7: Close the browser.

try:
    driver.quit()
    print("Browser closed.")
except NameError:
    print("No driver object exists yet.")


## Reflection

Example points to discuss:

1. Clicking, typing, waiting, scrolling, and screenshots are browser-automation actions that were not part of the static `requests` workflow.
2. `#query-box`, `#run-filter`, `#create-result`, `#message`, and `#evidence-end` are id selectors.
3. `.result-card` is a class selector.
4. `result_title` is visible text; `data-record-id` and `data-keyword` are hidden attributes read from the DOM.
5. The screenshot preserves visual browser state after interaction; the CSV preserves extracted structured records.
